# Train the Sentinel-1 flood segmentation U-Net (Phase 1, model C)

Runs on a free Colab T4. Fetches the canonical Sen1Floods11 files straight
from the **public GCS bucket** `sen1floods11` (no auth, no giant tar — the
old HF mirror now 401s anonymously), keeps only the **India event**
(2016 Assam): 467 weakly-labeled chips (train) + 68 hand-labeled chips
(validation) = 535 chips, ~0.9 GB. Trains the U-Net with
`ml/sar/train_unet.py`, reports an honest held-out val IoU/Dice on the
human-QC labels, and zips the artifacts for download into the repo's
`ml/artifacts/sar_unet/`.

**Setup:** put the repo's `ml/sar/` and `ml/requirements-ml.txt` on the Colab
runtime — either `git clone` (if you have a remote) or upload the `ml` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install torch segmentation-models-pytorch rasterio

In [ ]:
%cd /content
# Either clone your remote or upload the ml folder. Example for a local upload:
import os
if not os.path.exists('/content/ml/sar/train_unet.py'):
    print('Upload the ml folder (File > Upload) then re-run this cell.')
    print('Or: !git clone <your-repo-url> && cd <repo>')

In [ ]:
# 1) Download ONLY the India event from the public GCS bucket
#    (https://storage.googleapis.com/sen1floods11) — no auth, no tar, ~0.9 GB:
#      WeakLabeled/S1Hand_India_<id>.tif + LabelHand (Otsu auto labels)  = train (467)
#      HandLabeled/S1Hand_India_<id>.tif + LabelHand (human QC labels)   = val (68)
#    Sen1Floods11 has a single India event (2016 Assam, 535 chips), so this
#    IS 'India only'.
!python ml/sar/download_sen1floods11.py --out /content/sen1floods11 --events India
!df -h /content | tail -1

In [ ]:
# 2) Sanity: show tile / label pairs available.
tifs = !ls /content/sen1floods11/WeakLabeled/S1Hand_*.tif /content/sen1floods11/HandLabeled/S1Hand_*.tif | head -20
tifs

In [ ]:
# 3) Train (T4: ~40-80 s/epoch @256). Quick-fit with --epochs 3 to confirm
#    the pipeline, then the real run with --epochs 35 (~25-45 min).
#    Split = the dataset's own geographic split: train on WeakLabeled (Otsu
#    auto labels), validate on HandLabeled (human QC labels), so the reported
#    val IoU is on chips/labels the model never saw (no leak). Labels are the
#    canonical encoding: 1 = water, -1 = no data (excluded from the loss).
#    Recipe hardening (run #1 diverged to all-NaN weights with plain BCE@lr1e-3):
#      --lr 3e-4          lower LR for the pretrained backbone
#      --grad-clip 1.0    cap gradient norm (anti-NaN)
#      --pos-weight 8.0   up-weight the sparse water class (~1% of weak labels)
#    If a run ever diverges the script exits WITHOUT overwriting good artifacts
#    and writes diverged.log (fail-soft).
!python ml/sar/train_unet.py --data-dir /content/sen1floods11 \
    --epochs 3 --size 256 --batch-size 8 --encoder resnet18 --lr 3e-4 --grad-clip 1.0 --pos-weight 8.0 --out /content/artifacts/sar_unet
# Real run - re-run with a higher epoch count:
# !python ml/sar/train_unet.py --data-dir /content/sen1floods11 \
#     --epochs 35 --size 256 --batch-size 8 --encoder resnet18 --lr 3e-4 --grad-clip 1.0 --pos-weight 8.0 --out /content/artifacts/sar_unet

In [ ]:
# 4) Report the measured numbers (this is what goes in the evidence sheet).
import json
meta = json.load(open('/content/artifacts/sar_unet/meta.json'))
print(f"val IoU = {meta['val_iou']:.4f}   val Dice = {meta['val_dice']:.4f}")
print(meta['note'])

In [ ]:
# 5) Download the artifacts, then drop them into ml/artifacts/sar_unet/ in
#    the repo (model.pt + meta.json). The backend picks them up automatically.
import shutil
shutil.make_archive('/content/sar_unet', 'zip', '/content/artifacts/sar_unet')
from google.colab import files
files.download('/content/sar_unet.zip')